In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Optional: Install kneed for automatic elbow detection
# pip install kneed
try:
    from kneed import KneeLocator
    KNEED_AVAILABLE = True
except ImportError:
    KNEED_AVAILABLE = False
    print("Note: Install 'kneed' for automatic elbow detection: pip install kneed")

class OptimalKMeans:
    """
    Complete K-means optimization class with multiple evaluation methods
    """
    
    def __init__(self, max_k=15, random_state=42):
        self.max_k = max_k
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.results = []
        self.best_model = None
        self.scaled_data = None
        
    def prepare_data(self, data, apply_pca=False, n_components=None):
        """Prepare and scale data for clustering"""
        print("🔧 Preparing data...")
        
        # Convert to numpy array if needed
        if isinstance(data, (list, pd.DataFrame)):
            data = np.array(data)
        
        # Scale the data
        self.scaled_data = self.scaler.fit_transform(data)
        
        # Apply PCA if requested
        if apply_pca and n_components:
            self.pca = PCA(n_components=n_components, random_state=self.random_state)
            self.scaled_data = self.pca.fit_transform(self.scaled_data)
            print(f"📊 Applied PCA: {data.shape[1]} → {self.scaled_data.shape[1]} dimensions")
            print(f"📈 Explained variance: {self.pca.explained_variance_ratio_.sum():.3f}")
        
        print(f"✅ Data prepared: {self.scaled_data.shape}")
        return self.scaled_data
    
    def elbow_method(self, plot=True):
        """Find optimal K using elbow method"""
        print("📈 Running Elbow Method...")
        
        wcss = []
        k_range = range(1, self.max_k + 1)
        
        for k in tqdm(k_range, desc="Elbow Method"):
            kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, 
                          random_state=self.random_state)
            kmeans.fit(self.scaled_data)
            wcss.append(kmeans.inertia_)
        
        # Auto-detect elbow if kneed is available
        optimal_k_elbow = None
        if KNEED_AVAILABLE:
            knee_locator = KneeLocator(k_range, wcss, curve='convex', direction='decreasing')
            optimal_k_elbow = knee_locator.knee
            print(f"🎯 Auto-detected elbow at K = {optimal_k_elbow}")
        
        if plot:
            plt.figure(figsize=(10, 6))
            plt.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
            plt.title('Elbow Method For Optimal K', fontsize=16)
            plt.xlabel('Number of Clusters (K)', fontsize=12)
            plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
            plt.grid(True, alpha=0.3)
            
            if optimal_k_elbow:
                plt.axvline(x=optimal_k_elbow, color='red', linestyle='--', 
                           label=f'Auto-detected elbow (K={optimal_k_elbow})')
                plt.legend()
            
            plt.tight_layout()
            plt.show()
        
        return wcss, optimal_k_elbow
    
    def silhouette_analysis(self, plot=True):
        """Find optimal K using silhouette analysis"""
        print("📊 Running Silhouette Analysis...")
        
        silhouette_scores = []
        k_range = range(2, self.max_k + 1)
        
        for k in tqdm(k_range, desc="Silhouette Analysis"):
            kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, 
                          random_state=self.random_state)
            cluster_labels = kmeans.fit_predict(self.scaled_data)
            silhouette_avg = silhouette_score(self.scaled_data, cluster_labels)
            silhouette_scores.append(silhouette_avg)
        
        optimal_k_sil = k_range[np.argmax(silhouette_scores)]
        best_sil_score = max(silhouette_scores)
        
        print(f"🎯 Best K by Silhouette: {optimal_k_sil} (Score: {best_sil_score:.3f})")
        
        if plot:
            plt.figure(figsize=(10, 6))
            plt.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
            plt.title('Silhouette Analysis For Optimal K', fontsize=16)
            plt.xlabel('Number of Clusters (K)', fontsize=12)
            plt.ylabel('Average Silhouette Score', fontsize=12)
            plt.grid(True, alpha=0.3)
            plt.axhline(y=0.3, color='green', linestyle=':', alpha=0.7, 
                       label='Good threshold (0.3)')
            plt.axvline(x=optimal_k_sil, color='red', linestyle='--', 
                       label=f'Optimal K = {optimal_k_sil}')
            plt.legend()
            plt.tight_layout()
            plt.show()
        
        return silhouette_scores, optimal_k_sil
    
    def comprehensive_evaluation(self):
        """Comprehensive evaluation using all metrics"""
        print("🔍 Running Comprehensive Evaluation...")
        
        self.results = []
        k_range = range(2, self.max_k + 1)
        
        print(f"{'K':>3} {'WCSS':>10} {'Silhouette':>12} {'Calinski-H':>12} {'Davies-B':>10}")
        print("-" * 55)
        
        for k in tqdm(k_range, desc="Comprehensive Evaluation"):
            # Fit K-means
            kmeans = KMeans(
                n_clusters=k, 
                init='k-means++',
                n_init=20,  # More runs for stability
                max_iter=300,
                random_state=self.random_state
            )
            cluster_labels = kmeans.fit_predict(self.scaled_data)
            
            # Calculate all metrics
            wcss = kmeans.inertia_
            sil_score = silhouette_score(self.scaled_data, cluster_labels)
            cal_score = calinski_harabasz_score(self.scaled_data, cluster_labels)
            dbi_score = davies_bouldin_score(self.scaled_data, cluster_labels)
            
            # Store results
            result = {
                'k': k,
                'wcss': wcss,
                'silhouette': sil_score,
                'calinski_harabasz': cal_score,
                'davies_bouldin': dbi_score,
                'kmeans_model': kmeans,
                'cluster_labels': cluster_labels
            }
            self.results.append(result)
            
            print(f"{k:>3} {wcss:>10.2f} {sil_score:>12.3f} {cal_score:>12.2f} {dbi_score:>10.3f}")
        
        return self.results
    
    def find_optimal_k(self):
        """Find optimal K using multiple criteria"""
        if not self.results:
            self.comprehensive_evaluation()
        
        # Find best K by different criteria
        best_silhouette = max(self.results, key=lambda x: x['silhouette'])
        best_calinski = max(self.results, key=lambda x: x['calinski_harabasz'])
        best_davies = min(self.results, key=lambda x: x['davies_bouldin'])
        
        # Composite score (normalized)
        sil_scores = [r['silhouette'] for r in self.results]
        cal_scores = [r['calinski_harabasz'] for r in self.results]
        dbi_scores = [r['davies_bouldin'] for r in self.results]
        
        # Normalize scores (0-1 range)
        sil_norm = [(s - min(sil_scores)) / (max(sil_scores) - min(sil_scores)) for s in sil_scores]
        cal_norm = [(s - min(cal_scores)) / (max(cal_scores) - min(cal_scores)) for s in cal_scores]
        dbi_norm = [1 - (s - min(dbi_scores)) / (max(dbi_scores) - min(dbi_scores)) for s in dbi_scores]
        
        # Calculate composite scores
        for i, result in enumerate(self.results):
            result['composite_score'] = (sil_norm[i] + cal_norm[i] + dbi_norm[i]) / 3
        
        best_composite = max(self.results, key=lambda x: x['composite_score'])
        
        print(f"\n🏆 OPTIMIZATION RESULTS:")
        print(f"{'Criterion':<20} {'K':<3} {'Score':<10}")
        print("-" * 35)
        print(f"{'Silhouette':<20} {best_silhouette['k']:<3} {best_silhouette['silhouette']:<10.3f}")
        print(f"{'Calinski-Harabasz':<20} {best_calinski['k']:<3} {best_calinski['calinski_harabasz']:<10.2f}")
        print(f"{'Davies-Bouldin':<20} {best_davies['k']:<3} {best_davies['davies_bouldin']:<10.3f}")
        print(f"{'Composite':<20} {best_composite['k']:<3} {best_composite['composite_score']:<10.3f}")
        
        # Recommend best overall
        self.best_model = best_composite
        print(f"\n🎯 RECOMMENDED: K = {self.best_model['k']} (Composite Score)")
        
        return self.best_model
    
    def plot_all_metrics(self):
        """Plot all evaluation metrics"""
        if not self.results:
            self.comprehensive_evaluation()
        
        k_values = [r['k'] for r in self.results]
        sil_scores = [r['silhouette'] for r in self.results]
        cal_scores = [r['calinski_harabasz'] for r in self.results]
        dbi_scores = [r['davies_bouldin'] for r in self.results]
        composite_scores = [r['composite_score'] for r in self.results]
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Silhouette Score
        axes[0,0].plot(k_values, sil_scores, 'bo-', linewidth=2)
        axes[0,0].set_title('Silhouette Score', fontsize=14)
        axes[0,0].set_xlabel('Number of Clusters (K)')
        axes[0,0].set_ylabel('Silhouette Score')
        axes[0,0].grid(True, alpha=0.3)
        axes[0,0].axhline(y=0.3, color='green', linestyle=':', alpha=0.7)
        
        # Calinski-Harabasz Score
        axes[0,1].plot(k_values, cal_scores, 'ro-', linewidth=2)
        axes[0,1].set_title('Calinski-Harabasz Score', fontsize=14)
        axes[0,1].set_xlabel('Number of Clusters (K)')
        axes[0,1].set_ylabel('Calinski-Harabasz Score')
        axes[0,1].grid(True, alpha=0.3)
        
        # Davies-Bouldin Score (lower is better)
        axes[1,0].plot(k_values, dbi_scores, 'go-', linewidth=2)
        axes[1,0].set_title('Davies-Bouldin Score (Lower is Better)', fontsize=14)
        axes[1,0].set_xlabel('Number of Clusters (K)')
        axes[1,0].set_ylabel('Davies-Bouldin Score')
        axes[1,0].grid(True, alpha=0.3)
        
        # Composite Score
        axes[1,1].plot(k_values, composite_scores, 'mo-', linewidth=2)
        axes[1,1].set_title('Composite Score', fontsize=14)
        axes[1,1].set_xlabel('Number of Clusters (K)')
        axes[1,1].set_ylabel('Composite Score')
        axes[1,1].grid(True, alpha=0.3)
        
        # Highlight best K
        if self.best_model:
            best_k = self.best_model['k']
            for ax in axes.flat:
                ax.axvline(x=best_k, color='red', linestyle='--', alpha=0.7)
        
        plt.tight_layout()
        plt.show()
    
    def get_final_model(self, k=None):
        """Get final optimized K-means model"""
        if k is None:
            if self.best_model is None:
                self.find_optimal_k()
            k = self.best_model['k']
        
        # Create final model with optimal parameters
        final_kmeans = KMeans(
            n_clusters=k,
            init='k-means++',
            n_init=30,  # More runs for final model
            max_iter=300,
            tol=1e-4,
            random_state=self.random_state
        )
        
        # Fit and get final results
        final_labels = final_kmeans.fit_predict(self.scaled_data)
        
        # Calculate final metrics
        final_sil = silhouette_score(self.scaled_data, final_labels)
        final_cal = calinski_harabasz_score(self.scaled_data, final_labels)
        final_dbi = davies_bouldin_score(self.scaled_data, final_labels)
        
        print(f"\n✅ FINAL K-MEANS MODEL:")
        print(f"Optimal K: {k}")
        print(f"Silhouette Score: {final_sil:.3f}")
        print(f"Calinski-Harabasz Score: {final_cal:.2f}")
        print(f"Davies-Bouldin Score: {final_dbi:.3f}")
        print(f"Total samples: {len(self.scaled_data)}")
        print(f"Cluster sizes: {np.bincount(final_labels)}")
        
        return final_kmeans, final_labels

# Convenience function for quick optimization
def find_optimal_kmeans(data, max_k=15, apply_pca=False, n_components=None, 
                       plot_results=True, random_state=42):
    """
    Quick function to find optimal K-means configuration
    
    Parameters:
    - data: Your input data (embeddings, features, etc.)
    - max_k: Maximum number of clusters to test
    - apply_pca: Whether to apply PCA for dimensionality reduction
    - n_components: Number of PCA components (if apply_pca=True)
    - plot_results: Whether to show plots
    - random_state: Random state for reproducibility
    
    Returns:
    - optimal_kmeans: Fitted K-means model with optimal K
    - cluster_labels: Cluster assignments
    - optimizer: OptimalKMeans object for detailed analysis
    """
    
    print("🚀 Starting K-means Optimization...")
    
    # Initialize optimizer
    optimizer = OptimalKMeans(max_k=max_k, random_state=random_state)
    
    # Prepare data
    optimizer.prepare_data(data, apply_pca=apply_pca, n_components=n_components)
    
    # Run all optimization methods
    if plot_results:
        optimizer.elbow_method(plot=True)
        optimizer.silhouette_analysis(plot=True)
    
    # Find optimal K
    best_config = optimizer.find_optimal_k()
    
    # Plot all metrics
    if plot_results:
        optimizer.plot_all_metrics()
    
    # Get final model
    optimal_kmeans, cluster_labels = optimizer.get_final_model()
    
    return optimal_kmeans, cluster_labels, optimizer

# Usage Examples
if __name__ == "__main__":
    # Example usage with your embedding data
    
    """
    # Method 1: Quick optimization
    optimal_model, labels, optimizer = find_optimal_kmeans(
        data=all_embeddings,  # Your embedding data
        max_k=15,
        apply_pca=True,       # Use PCA for high-dimensional data
        n_components=20,      # Reduce to 20 dimensions
        plot_results=True
    )
    
    # Method 2: Step-by-step approach
    optimizer = OptimalKMeans(max_k=15)
    
    # Prepare data
    scaled_data = optimizer.prepare_data(all_embeddings, apply_pca=True, n_components=20)
    
    # Run different methods
    wcss, elbow_k = optimizer.elbow_method()
    sil_scores, sil_k = optimizer.silhouette_analysis()
    
    # Comprehensive evaluation
    results = optimizer.comprehensive_evaluation()
    
    # Find optimal K
    best_model = optimizer.find_optimal_k()
    
    # Plot all metrics
    optimizer.plot_all_metrics()
    
    # Get final model
    final_model, final_labels = optimizer.get_final_model()
    
    print("🎉 Optimization Complete!")
    """
    


In [ ]:
# For your embedding data
optimal_model, cluster_labels, optimizer = find_optimal_kmeans(
    data=all_embeddings,
    max_k=15,
    apply_pca=True,
    n_components=20,
    plot_results=True
)

# Access results
print(f"Optimal number of clusters: {optimal_model.n_clusters}")
print(f"Cluster assignments: {cluster_labels}")

# Get detailed results
results_df = pd.DataFrame(optimizer.results)
print(results_df[['k', 'silhouette', 'calinski_harabasz', 'davies_bouldin']])
